# 🎨 ComfyUI en Google Colab — Drive, modelos e imagen a vídeo

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/I-s-ax/Comfyui_en_colab/blob/main/comfyui_colab_drive.ipynb)

Notebook independiente, preparado para usar ComfyUI desde Android con almacenamiento en Google Drive y formularios de Colab.

**Orden recomendado:** selecciona una GPU en *Entorno de ejecución → Cambiar tipo de entorno*; ejecuta las celdas de arriba hacia abajo; descarga un checkpoint/LoRA y después inicia ComfyUI.

Los campos `#@param` aparecen como formularios. Para generar vídeo también necesitas un modelo de vídeo y un workflow compatible. Este notebook está inspirado en la organización de otros proyectos públicos, pero su código es propio.

## 1. Conectar Google Drive

In [ ]:
#@title 🔌 Conectar Google Drive
from google.colab import drive
from pathlib import Path

MONTAR_DRIVE = True #@param {type:"boolean"}
NOMBRE_CARPETA = "ComfyUI" #@param {type:"string"}

if MONTAR_DRIVE:
    drive.mount("/content/drive")
    WORKSPACE = str(Path("/content/drive/MyDrive") / NOMBRE_CARPETA)
else:
    WORKSPACE = "/content/ComfyUI"

Path(WORKSPACE).mkdir(parents=True, exist_ok=True)
print("Espacio de trabajo:", WORKSPACE)

## 2. Instalar o actualizar ComfyUI

In [ ]:
#@title 🧰 Instalar / actualizar ComfyUI
import subprocess
from pathlib import Path

ACTUALIZAR_COMFYUI = True #@param {type:"boolean"}
repo = Path(WORKSPACE)

if (repo / ".git").exists():
    if ACTUALIZAR_COMFYUI:
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    if any(repo.iterdir()):
        raise RuntimeError(f"{repo} ya existe y no parece una instalación Git. Cambia NOMBRE_CARPETA en la primera celda.")
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", str(repo)], check=True)

subprocess.run(["python", "-m", "pip", "install", "-r", str(repo / "requirements.txt")], check=True)
for name in ["checkpoints", "vae", "loras", "controlnet", "upscale_models", "diffusion_models", "text_encoders", "clip_vision"]:
    (repo / "models" / name).mkdir(parents=True, exist_ok=True)
print("ComfyUI preparado en:", repo)

## 3. Descargar un modelo
Pega un enlace de descarga de Civitai o Hugging Face. Si Civitai requiere token, guárdalo en Colab Secrets con el nombre `CIVITAI_API_TOKEN` o introdúcelo en el campo manual. No compartas el token.

In [ ]:
#@title 📥 Descargar modelo
URL_MODELO = "" #@param {type:"string"}
CARPETA_DESTINO = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models", "diffusion_models", "text_encoders", "clip_vision"]
NOMBRE_ARCHIVO = "" #@param {type:"string"}
TOKEN_CIVITAI_MANUAL = "" #@param {type:"string"}

import os, re, time, requests
from pathlib import Path
from urllib.parse import urlparse, unquote

if not URL_MODELO.strip():
    raise ValueError("Pega un enlace en URL_MODELO.")

token = TOKEN_CIVITAI_MANUAL.strip()
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("CIVITAI_API_TOKEN") or ""
    except Exception:
        token = ""

original_url = URL_MODELO.strip()
url = original_url
if "civitai.com" in urlparse(url).netloc.lower() and token:
    url += ("&" if "?" in url else "?") + "token=" + token

dest = Path(WORKSPACE) / "models" / CARPETA_DESTINO
dest.mkdir(parents=True, exist_ok=True)
filename = NOMBRE_ARCHIVO.strip() or unquote(Path(urlparse(original_url).path).name)
if not filename or filename.lower() in ("download", "models"):
    filename = "modelo_descargado.safetensors"
target = dest / Path(filename).name

if target.exists() and target.stat().st_size:
    print("Ya existe:", target, f"({target.stat().st_size/1e9:.2f} GB)")
else:
    error = None
    for attempt in range(3):
        temp = target.with_name(target.name + ".part")
        try:
            with requests.get(url, stream=True, allow_redirects=True, timeout=(30, 120),
                              headers={"User-Agent": "ComfyUI-Colab-Downloader/1.0"}) as r:
                if r.status_code != 200:
                    raise RuntimeError(f"HTTP {r.status_code}; revisa enlace, permisos y token.")
                if "text/html" in r.headers.get("Content-Type", "").lower():
                    raise RuntimeError("El servidor devolvió HTML, no el archivo del modelo.")
                cd = r.headers.get("Content-Disposition", "")
                if not NOMBRE_ARCHIVO.strip():
                    m = re.search(r'filename="?([^";]+)', cd, re.I)
                    if m:
                        target = dest / Path(unquote(m.group(1))).name
                        temp = target.with_name(target.name + ".part")
                total = int(r.headers.get("Content-Length", "0") or 0)
                got = 0
                with open(temp, "wb") as f:
                    for chunk in r.iter_content(8 * 1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            got += len(chunk)
                            if total:
                                print(f"Descargado {got/1e9:.2f}/{total/1e9:.2f} GB", end="\r")
                if total and got != total:
                    raise RuntimeError(f"Descarga incompleta ({got}/{total} bytes).")
                with open(temp, "rb") as f:
                    head = f.read(500).lower()
                if b"<html" in head or b"<!doctype" in head:
                    raise RuntimeError("El contenido parece HTML, no un modelo.")
                os.replace(temp, target)
                print(f"\nDescarga completada: {target} ({got/1e9:.2f} GB)")
                error = None
                break
        except Exception as e:
            error = e
            if temp.exists():
                temp.unlink()
            print(f"Intento {attempt+1}/3: {e}")
            if attempt < 2:
                time.sleep(2 * (attempt + 1))
    if error:
        raise RuntimeError(f"No se pudo descargar: {error}") from error
print("Carpeta:", dest)

## 4. Listar modelos

In [ ]:
#@title 🔎 Listar archivos de modelos
MODELOS_A_LISTAR = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models", "diffusion_models", "text_encoders", "clip_vision"]

from pathlib import Path
folder = Path(WORKSPACE) / "models" / MODELOS_A_LISTAR
folder.mkdir(parents=True, exist_ok=True)
files = sorted(p for p in folder.iterdir() if p.is_file())
print("Carpeta:", folder)
if not files:
    print("No hay archivos en esta carpeta.")
for p in files:
    print(f"{p.name} — {p.stat().st_size/1e9:.2f} GB")

## 5. Iniciar ComfyUI y abrir el enlace

In [ ]:
#@title 🚀 Iniciar ComfyUI y mostrar enlace (LocalTunnel primero)
import subprocess, time, re, sys, urllib.request, shutil, threading, queue
from pathlib import Path

repo = Path(WORKSPACE)
if not (repo / "main.py").exists():
    raise FileNotFoundError("No encuentro main.py. Ejecuta primero la instalación.")

def comprobar_comfyui():
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188", timeout=5) as respuesta:
            return respuesta.status
    except Exception:
        return None

# Evita duplicar ComfyUI: primero comprueba el servidor y luego los procesos.
estado = comprobar_comfyui()
comfy = None
if estado:
    print(f"✅ ComfyUI ya estaba activo (HTTP {estado}); se reutilizará el proceso.", flush=True)
else:
    log_path = Path("/content/comfyui.log")
    salida = queue.Queue()
    log = open(log_path, "w", buffering=1)
    comfy = subprocess.Popen(
        [sys.executable, "-u", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
        cwd=str(repo), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )

    def leer_log_comfyui():
        try:
            for linea in comfy.stdout:
                log.write(linea)
                salida.put(linea)
        finally:
            salida.put(None)

    threading.Thread(target=leer_log_comfyui, daemon=True).start()
    print("⏳ Iniciando ComfyUI. Ahora se mostrarán los mensajes reales de carga.", flush=True)
    print(f"📝 El registro también se guarda en {log_path}", flush=True)
    inicio = time.time()
    limite_segundos = 360  # hasta 6 minutos para importar nodos personalizados
    ultimo_aviso = inicio
    fin_salida = False

    while time.time() - inicio < limite_segundos:
        if comfy.poll() is not None:
            break

        try:
            linea = salida.get(timeout=2)
            if linea is None:
                fin_salida = True
            else:
                print(linea, end="", flush=True)
        except queue.Empty:
            pass

        estado = comprobar_comfyui()
        if estado:
            print(f"\n✅ ComfyUI responde correctamente (HTTP {estado}).", flush=True)
            break

        if time.time() - ultimo_aviso >= 30:
            transcurrido = int(time.time() - inicio)
            print(f"… ComfyUI sigue cargando ({transcurrido}s). Revisa los mensajes anteriores.", flush=True)
            ultimo_aviso = time.time()

    if not estado:
        if comfy.poll() is None:
            comfy.terminate()
            try:
                comfy.wait(timeout=10)
            except subprocess.TimeoutExpired:
                comfy.kill()
        time.sleep(0.5)
        log.close()
        detalle = log_path.read_text(errors="ignore")[-6000:]
        print("\n===== FINAL DEL REGISTRO DE COMFYUI =====", flush=True)
        print(detalle or "(El registro está vacío)", flush=True)
        raise RuntimeError(
            "ComfyUI no respondió en 6 minutos. Se muestra arriba el registro para identificar "
            "el módulo o dependencia que detiene el inicio. No vuelvas a ejecutar la celda de túnel "
            "hasta revisar ese mensaje."
        )

    print("✅ ComfyUI está listo; ahora se preparará el enlace público.", flush=True)

def lanzar_localtunnel():
    print("\n📦 Preparando LocalTunnel (método principal)...", flush=True)
    if not shutil.which("npm"):
        raise RuntimeError("No se encontró npm en este entorno de Colab.")
    if not shutil.which("lt"):
        instalacion = subprocess.run(
            ["npm", "install", "-g", "localtunnel"],
            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
        )
        print(instalacion.stdout[-2500:], flush=True)
        if instalacion.returncode != 0 or not shutil.which("lt"):
            raise RuntimeError("No se pudo instalar LocalTunnel.")

    print("\n🔑 Contraseña que puede pedir LocalTunnel:", flush=True)
    try:
        solicitud = urllib.request.Request(
            "https://loca.lt/mytunnelpassword",
            headers={"User-Agent": "Mozilla/5.0"}
        )
        with urllib.request.urlopen(solicitud, timeout=15) as respuesta:
            print(respuesta.read().decode().strip(), flush=True)
    except Exception as e:
        print(f"No se pudo consultar automáticamente la contraseña: {e}", flush=True)
        print("Si la página la solicita, vuelve a ejecutar la consulta en otra celda.", flush=True)

    print("\n🌐 Iniciando LocalTunnel. Espera el enlace https://....loca.lt", flush=True)
    proceso = subprocess.Popen(
        ["lt", "--port", "8188"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    salida = queue.Queue()
    def leer_salida():
        for linea in proceso.stdout:
            salida.put(linea)
        salida.put(None)
    threading.Thread(target=leer_salida, daemon=True).start()

    limite = time.time() + 60
    url_lt = None
    while time.time() < limite and proceso.poll() is None:
        try:
            linea = salida.get(timeout=1)
        except queue.Empty:
            continue
        if linea is None:
            break
        print(linea, end="", flush=True)
        m = re.search(r"https://[A-Za-z0-9.-]+\.loca\.lt", linea)
        if m:
            url_lt = m.group(0)
            print(f"\n✅ Enlace de ComfyUI: {url_lt}", flush=True)
            print("Deja esta celda ejecutándose mientras uses ComfyUI.", flush=True)
            # Mantener vivo el proceso del túnel y seguir mostrando su salida.
            while proceso.poll() is None:
                try:
                    linea = salida.get(timeout=1)
                    if linea is None:
                        break
                    print(linea, end="", flush=True)
                except queue.Empty:
                    pass
            return True

    if proceso.poll() is None:
        proceso.terminate()
        try:
            proceso.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proceso.kill()
    return False

def lanzar_cloudflare():
    print("\n⚠️ LocalTunnel no generó un enlace. Probando Cloudflare como alternativa...", flush=True)
    cloudflared = "/usr/local/bin/cloudflared"
    if not Path(cloudflared).exists():
        try:
            subprocess.run(
                "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
                shell=True, check=True, timeout=60
            )
        except Exception as e:
            raise RuntimeError(f"No se pudo instalar cloudflared: {e}") from e

    log_path = Path("/content/cloudflared.log")
    with open(log_path, "w") as log:
        proceso = subprocess.Popen(
            [cloudflared, "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
            stdout=log, stderr=subprocess.STDOUT
        )
        url_cf = None
        for _ in range(45):
            if proceso.poll() is not None:
                break
            time.sleep(2)
            contenido = log_path.read_text(errors="ignore")
            m = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", contenido)
            if m:
                url_cf = m.group(0)
                break
    if url_cf:
        print(f"🌐 Enlace alternativo de Cloudflare: {url_cf}", flush=True)
        print("Si este enlace muestra un error de acceso, detén la celda y vuelve a intentar: LocalTunnel es el método principal.", flush=True)
        print("Deja esta celda ejecutándose mientras uses ComfyUI.", flush=True)
        while proceso.poll() is None:
            time.sleep(2)
    else:
        if proceso.poll() is None:
            proceso.terminate()
        print("No se pudo crear un túnel. Último registro:", flush=True)
        print(log_path.read_text(errors="ignore")[-3000:], flush=True)
        raise RuntimeError("No se pudo obtener un enlace. Revisa el registro anterior.")

# LocalTunnel evita el problema de acceso que ocurrió con el enlace trycloudflare.
# Si LocalTunnel no puede crear el enlace, se intenta Cloudflare automáticamente.
if not lanzar_localtunnel():
    lanzar_cloudflare()


## Solución de problemas
- **403 de Civitai:** verifica el token, los permisos y que el enlace corresponda al archivo correcto.
- **404:** comprueba el ID del modelo y el `fileId`; son identificadores distintos.
- **El modelo no aparece:** confirma la carpeta de destino, que el archivo no termine en `.part`, y refresca o reinicia ComfyUI.
- **T4 sin memoria:** baja la resolución y el batch size; evita cargar varios modelos grandes a la vez.
- **Drive lento:** el almacenamiento temporal de Colab se borra al finalizar la sesión.
- Checkpoints, LoRA, VAE y modelos de vídeo son tipos distintos y necesitan workflows compatibles.

El enlace de Cloudflare es temporal y público. No uses una sesión expuesta para contenido sensible sin añadir protección.